# Lexical POS Classification Experiments

Reproducibility notebook for **A Context-Free Framework for Evaluating Morphological Generalization in Lexical POS Classification**.

This notebook contains only the analyses required to reproduce the reported experiments: dataset audit and label construction, deterministic standard splits, the five manuscript model comparisons, uncertainty estimation and paired tests, suffix hold-out evaluation, convergence diagnostics, configuration export, and the primary suffix hold-out figure.

**Data:** obtain the original `words_pos.csv` from the source described in `../data/README.md` and place it in the repository's `data/` directory before running this notebook.

**Primary evaluation:** Accuracy and Macro-F1 for the standard tasks; supported-class Macro-F1 for suffix hold-out evaluation.

**Model labels are aligned with the manuscript tables:** Majority-class baseline, Rule-based suffix, Statistical suffix, Char N-gram, and BiLSTM.


In [ ]:
# ============================================================
# 0. Imports, paths, and global configuration
# ============================================================
from pathlib import Path
import copy
import json
import math
import os
import platform
import random
import sys
import time
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import __version__ as sklearn_version
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import murmurhash3_32

from scipy.stats import beta

warnings.filterwarnings("ignore")

# ---- Repository-relative paths ----
# Works whether Jupyter is launched from the repository root
# or from the notebooks/ directory.
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_PATH = REPO_ROOT / "data" / "words_pos.csv"
OUT_DIR = REPO_ROOT / "results" / "generated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Reproducibility ----
SPLIT_SEED = 42
PRIMARY_SEED = 42                     # single prediction set for paired test / plots

# ---- Evaluation ----
BOOTSTRAP_REPS = 2000
PERMUTATION_REPS = 2000
ALPHA = 0.05

# ---- Fine-grained task ----
MIN_COUNT_FINE = 50

# ---- Hash model ----
HASH_N_FEATURES = 2**18
HASH_NGRAM_RANGE = (2, 5)
SGD_MAX_EPOCHS = 20
SGD_PATIENCE = 3
SGD_BATCH_SIZE = 50_000
SGD_ALPHA = 1e-4
SGD_LOSS = "log_loss"
SGD_PENALTY = "l2"
SGD_LEARNING_RATE = "optimal"

# ---- BiLSTM ----
BILSTM_MAX_LEN = 30
BILSTM_EMBED_DIM = 64
BILSTM_HIDDEN_DIM = 128
BILSTM_BATCH_SIZE = 256
BILSTM_LR = 1e-3
BILSTM_MAX_EPOCHS = 20
BILSTM_PATIENCE = 3

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn_version)
print("Platform:", platform.platform())

In [ ]:
# ============================================================
# 1. Global seed helper
# ============================================================
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        # Deterministic behavior where supported
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_global_seed(PRIMARY_SEED)

## 2. Dataset source and audit

The source file is the Kaggle 370k English words POS resource using Penn Treebank-style tags; it is not the Penn Treebank corpus itself. The audit below records row counts, normalized lexical-form uniqueness, duplicate pairs, missing values, and non-alphabetic entries.


In [ ]:
# ============================================================
# 2. Load and audit source data
# ============================================================

raw = pd.read_csv(DATA_PATH, usecols=["word", "pos_tag"])

# ------------------------------------------------------------
# Basic audit of the original file
# ------------------------------------------------------------
audit = {}

audit["raw_rows"] = len(raw)
audit["missing_word"] = int(raw["word"].isna().sum())
audit["missing_pos_tag"] = int(raw["pos_tag"].isna().sum())

# Create normalized versions WITHOUT overwriting original columns
tmp = raw.copy()

tmp["word_norm"] = (
    tmp["word"]
    .astype("string")
    .str.strip()
    .str.lower()
)

tmp["pos_tag_norm"] = (
    tmp["pos_tag"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# ------------------------------------------------------------
# Dataset audit
# ------------------------------------------------------------
audit["unique_raw_words"] = int(
    raw["word"].nunique(dropna=True)
)

audit["unique_normalized_words"] = int(
    tmp["word_norm"].nunique(dropna=True)
)

audit["duplicate_word_tag_pairs"] = int(
    tmp.duplicated(
        subset=["word_norm", "pos_tag_norm"]
    ).sum()
)

word_label_counts = (
    tmp.dropna(subset=["word_norm", "pos_tag_norm"])
       .groupby("word_norm")["pos_tag_norm"]
       .nunique()
)

audit["words_with_multiple_source_labels"] = int(
    (word_label_counts > 1).sum()
)

audit["max_labels_per_word"] = (
    int(word_label_counts.max())
    if len(word_label_counts) > 0
    else 0
)

audit["nonalphabetic_words"] = int(
    (
        ~tmp["word_norm"]
        .fillna("")
        .str.fullmatch(r"[a-z]+")
    ).sum()
)

# ------------------------------------------------------------
# Display and save audit
# ------------------------------------------------------------
audit_df = pd.DataFrame({
    "Property": list(audit.keys()),
    "Value": list(audit.values())
})

display(audit_df)

OUT_DIR.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(
    OUT_DIR / "dataset_audit.csv",
    index=False
)

# ============================================================
# Create cleaned analysis dataset
# ============================================================

# IMPORTANT:
# Build a NEW dataframe instead of renaming columns in tmp.
df = pd.DataFrame({
    "word": tmp["word_norm"],
    "pos_tag": tmp["pos_tag_norm"]
})

# Remove missing and empty entries
df = df.dropna(
    subset=["word", "pos_tag"]
).copy()

df = df[
    (df["word"] != "") &
    (df["pos_tag"] != "")
].copy()

# Retain alphabetic lexical forms only
df = df[
    df["word"].str.fullmatch(r"[a-z]+", na=False)
].copy()

# Remove duplicate word-tag pairs, if any
df = (
    df.drop_duplicates(
        subset=["word", "pos_tag"]
    )
    .reset_index(drop=True)
)

# ============================================================
# Lexical uniqueness check
# ============================================================

duplicate_words = df["word"].duplicated(
    keep=False
)

if duplicate_words.any():

    ambiguous_words = (
        df.loc[duplicate_words]
        .sort_values("word")
    )

    print(
        f"WARNING: {ambiguous_words['word'].nunique()} "
        "normalized words occur with more than one row."
    )

    display(
        ambiguous_words.head(20)
    )

else:
    print(
        "No duplicate normalized word forms were found."
    )

# Because your current source dataset is expected to contain
# one row per lexical form, we can assert uniqueness.
assert df["word"].is_unique, (
    "Normalized word forms are not unique. "
    "Use group-based splitting or resolve lexical duplicates "
    "before proceeding."
)

# ============================================================
# Final dataset summary
# ============================================================

print("\nFinal analysis rows:",
      len(df))

print("Unique normalized words:",
      df["word"].nunique())

print("Number of POS tags:",
      df["pos_tag"].nunique())

print("\nPOS distribution:")

display(
    df["pos_tag"]
    .value_counts()
    .rename_axis("POS tag")
    .to_frame("count")
)

### Interpretation of lexical ambiguity

The source file assigns exactly one reference POS label to each normalized word form if `words_with_multiple_source_labels == 0`. This prevents **within-file lexical leakage**, but it does **not** imply that English lexical items are linguistically unambiguous. The manuscript should acknowledge that context-dependent ambiguity is not represented by this automatically prepared lexical resource.

In [ ]:
# ============================================================
# 3. Coarse and fine label construction
# ============================================================
PTB_TO_COARSE = {
    "NN": "NOUN", "NNS": "NOUN", "NNP": "NOUN", "NNPS": "NOUN",
    "VB": "VERB", "VBD": "VERB", "VBG": "VERB", "VBN": "VERB",
    "VBP": "VERB", "VBZ": "VERB",
    "JJ": "ADJ", "JJR": "ADJ", "JJS": "ADJ",
    "RB": "ADV", "RBR": "ADV", "RBS": "ADV", "WRB": "ADV",
    "DT": "FUNCTION", "IN": "FUNCTION", "CC": "FUNCTION",
    "PRP": "FUNCTION", "PRP$": "FUNCTION", "WDT": "FUNCTION",
    "WP": "FUNCTION", "WP$": "FUNCTION", "TO": "FUNCTION",
    "MD": "FUNCTION", "CD": "FUNCTION"
}

df["coarse_tag"] = df["pos_tag"].map(PTB_TO_COARSE)
df_coarse = df.dropna(subset=["coarse_tag"]).copy()

fine_counts = df["pos_tag"].value_counts()
valid_fine_tags = sorted(fine_counts[fine_counts >= MIN_COUNT_FINE].index.tolist())
df_fine = df[df["pos_tag"].isin(valid_fine_tags)].copy()

print("Coarse counts:")
display(df_coarse["coarse_tag"].value_counts().to_frame("count"))
print("Fine tags retained:", valid_fine_tags)
display(df_fine["pos_tag"].value_counts().to_frame("count"))

In [ ]:
# ============================================================
# 4. Stratified 70/15/15 split + explicit lexical-overlap audit
# ============================================================
def stratified_split(data, label_col, test_size=0.15, val_size=0.15, random_state=42):
    train_val, test = train_test_split(
        data,
        test_size=test_size,
        stratify=data[label_col],
        random_state=random_state
    )
    relative_val_size = val_size / (1 - test_size)
    train, val = train_test_split(
        train_val,
        test_size=relative_val_size,
        stratify=train_val[label_col],
        random_state=random_state
    )
    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True)
    )

def overlap_report(train, val, test):
    tr, va, te = set(train["word"]), set(val["word"]), set(test["word"])
    return {
        "train_val_overlap": len(tr & va),
        "train_test_overlap": len(tr & te),
        "val_test_overlap": len(va & te),
    }

train_c, val_c, test_c = stratified_split(
    df_coarse, "coarse_tag", random_state=SPLIT_SEED
)
train_f, val_f, test_f = stratified_split(
    df_fine, "pos_tag", random_state=SPLIT_SEED
)

coarse_overlap = overlap_report(train_c, val_c, test_c)
fine_overlap = overlap_report(train_f, val_f, test_f)

print("Coarse split sizes:", len(train_c), len(val_c), len(test_c))
print("Coarse lexical overlap:", coarse_overlap)
print("Fine split sizes:", len(train_f), len(val_f), len(test_f))
print("Fine lexical overlap:", fine_overlap)

assert all(v == 0 for v in coarse_overlap.values())
assert all(v == 0 for v in fine_overlap.values())

# Save exact split identifiers for reproducibility
for name, frame in [
    ("coarse_train", train_c), ("coarse_val", val_c), ("coarse_test", test_c),
    ("fine_train", train_f), ("fine_val", val_f), ("fine_test", test_f),
]:
    frame[["word", "pos_tag"] + (["coarse_tag"] if "coarse" in name else [])].to_csv(
        OUT_DIR / f"{name}.csv", index=False
    )

## 5. Evaluation helpers

Macro-F1 is the primary metric, but its uncertainty is important because the coarse FUNCTION test subset is very small. The notebook therefore reports:
- point estimates,
- stratified bootstrap 95% confidence intervals,
- exact Clopper–Pearson intervals for FUNCTION recall,
- paired approximate-randomization tests for adjacent models.

In [ ]:
# ============================================================
# 5. Metrics, bootstrap CIs, exact interval, paired tests
# ============================================================
def metric_dict(y_true, y_pred, labels=None):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
    }

def stratified_bootstrap_ci(
    y_true, y_pred, metric="macro_f1", labels=None,
    n_boot=2000, seed=123, alpha=0.05
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(seed)

    strata = {
        lab: np.where(y_true == lab)[0]
        for lab in np.unique(y_true)
    }

    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([
            rng.choice(indices, size=len(indices), replace=True)
            for indices in strata.values()
        ])
        if metric == "accuracy":
            val = accuracy_score(y_true[idx], y_pred[idx])
        elif metric == "macro_f1":
            val = f1_score(
                y_true[idx], y_pred[idx],
                labels=labels, average="macro", zero_division=0
            )
        else:
            raise ValueError(metric)
        vals.append(val)

    lo, hi = np.quantile(vals, [alpha/2, 1-alpha/2])
    return float(lo), float(hi)

def clopper_pearson(k, n, alpha=0.05):
    if n == 0:
        return (np.nan, np.nan)
    lo = 0.0 if k == 0 else beta.ppf(alpha/2, k, n-k+1)
    hi = 1.0 if k == n else beta.ppf(1-alpha/2, k+1, n-k)
    return float(lo), float(hi)

def paired_macro_f1_randomization_test(
    y_true, pred_a, pred_b, labels=None,
    n_perm=2000, seed=123
):
    """Approximate paired randomization test for delta Macro-F1."""
    y_true = np.asarray(y_true)
    a = np.asarray(pred_a).copy()
    b = np.asarray(pred_b).copy()

    obs = (
        f1_score(y_true, b, labels=labels, average="macro", zero_division=0)
        - f1_score(y_true, a, labels=labels, average="macro", zero_division=0)
    )

    rng = np.random.default_rng(seed)
    more_extreme = 0

    for _ in range(n_perm):
        swap = rng.random(len(y_true)) < 0.5
        pa = a.copy()
        pb = b.copy()
        pa[swap], pb[swap] = b[swap], a[swap]

        d = (
            f1_score(y_true, pb, labels=labels, average="macro", zero_division=0)
            - f1_score(y_true, pa, labels=labels, average="macro", zero_division=0)
        )
        if abs(d) >= abs(obs):
            more_extreme += 1

    p = (more_extreme + 1) / (n_perm + 1)
    return float(obs), float(p)

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p_values[idx]
        running = max(running, val)
        adj[idx] = min(running, 1.0)
    return adj

## 6. Baselines and rule inventories

The revised analysis includes a majority-class baseline.

For suffix hold-out evaluation, two rule conditions are defined:
- **Full-rule reference**: retains all linguistic rules; this is an external-knowledge reference only.
- **Masked-rule baseline**: removes the rule corresponding to the held-out suffix family and is the fair comparison for morphological generalization.

The manuscript should not interpret the full-rule condition as evidence of learned generalization.

In [ ]:
# ============================================================
# 6. Majority and rule-based baselines
# ============================================================
COARSE_RULES = [
    ("tion", "NOUN"), ("sion", "NOUN"), ("ment", "NOUN"),
    ("ness", "NOUN"), ("ity", "NOUN"), ("ism", "NOUN"),
    ("ist", "NOUN"),
    ("ly", "ADV"),
    ("ous", "ADJ"), ("ive", "ADJ"), ("able", "ADJ"),
    ("ible", "ADJ"), ("al", "ADJ"), ("ic", "ADJ"),
    ("ful", "ADJ"), ("less", "ADJ"), ("ish", "ADJ"),
    ("ing", "VERB"), ("ed", "VERB"), ("en", "VERB"),
    ("ize", "VERB"), ("ise", "VERB"),
]

# Explicit fine-grained rules. These are intentionally simple and
# should be documented as a deterministic linguistic baseline.
FINE_RULES = [
    ("tion", "NN"), ("sion", "NN"), ("ment", "NN"),
    ("ness", "NN"), ("ity", "NN"), ("ism", "NN"), ("ist", "NN"),
    ("ly", "RB"),
    ("ous", "JJ"), ("ive", "JJ"), ("able", "JJ"), ("ible", "JJ"),
    ("al", "JJ"), ("ic", "JJ"), ("ful", "JJ"), ("less", "JJ"), ("ish", "JJ"),
    ("ing", "VBG"), ("ed", "VBD"), ("en", "VBN"),
    ("ize", "VB"), ("ise", "VB"),
]

def majority_predict(train_labels, n):
    majority = Counter(train_labels).most_common(1)[0][0]
    return np.repeat(majority, n)

def rule_predict(word, rules, default_label, masked_suffixes=None):
    masked_suffixes = set(masked_suffixes or [])
    for suffix, label in sorted(rules, key=lambda x: len(x[0]), reverse=True):
        if suffix in masked_suffixes:
            continue
        if word.endswith(suffix):
            return label
    return default_label

def rule_predict_many(words, rules, default_label, masked_suffixes=None):
    return [
        rule_predict(w, rules, default_label, masked_suffixes)
        for w in words
    ]

In [ ]:
# ============================================================
# 7. Statistical suffix model
# ============================================================
def train_suffix_model(train_df, label_col, max_suffix_len=5):
    suffix_stats = defaultdict(Counter)
    label_prior = Counter(train_df[label_col])

    for word, label in zip(train_df["word"], train_df[label_col]):
        for n in range(1, max_suffix_len + 1):
            if len(word) >= n:
                suffix_stats[word[-n:]][label] += 1

    return suffix_stats, label_prior

def predict_suffix_model(word, suffix_stats, label_prior, max_suffix_len=5):
    for n in range(max_suffix_len, 0, -1):
        if len(word) >= n:
            suffix = word[-n:]
            if suffix in suffix_stats:
                return suffix_stats[suffix].most_common(1)[0][0]
    return label_prior.most_common(1)[0][0]

def suffix_predict_many(train_df, test_words, label_col, max_suffix_len=5):
    stats, prior = train_suffix_model(train_df, label_col, max_suffix_len)
    return [
        predict_suffix_model(w, stats, prior, max_suffix_len)
        for w in test_words
    ]

## 8. Character-hash collision audit

This section reports the hashing dimension and observed collision rate by enumerating distinct character n-grams in the training corpus and applying scikit-learn's MurmurHash function.


In [ ]:
# ============================================================
# 8. Hash collision audit
# ============================================================
def hash_collision_audit(words, n_features=HASH_N_FEATURES, ngram_range=HASH_NGRAM_RANGE):
    vec = HashingVectorizer(
        analyzer="char",
        ngram_range=ngram_range,
        n_features=n_features,
        alternate_sign=False,
        norm="l2"
    )
    analyzer = vec.build_analyzer()

    unique_ngrams = set()
    for w in words:
        unique_ngrams.update(analyzer(w))

    bins = {
        murmurhash3_32(ng, seed=0, positive=True) % n_features
        for ng in unique_ngrams
    }

    n_unique = len(unique_ngrams)
    n_bins = len(bins)
    collisions = n_unique - n_bins
    collision_rate = collisions / n_unique if n_unique else 0.0

    return {
        "n_features": int(n_features),
        "unique_character_ngrams": int(n_unique),
        "occupied_hash_bins": int(n_bins),
        "collisions": int(collisions),
        "collision_rate": float(collision_rate),
    }

collision_info = hash_collision_audit(train_c["word"].tolist())
print(collision_info)

with open(OUT_DIR / "hash_collision_audit.json", "w") as f:
    json.dump(collision_info, f, indent=2)

## 9. Character n-gram + SGD classifier

The implementation explicitly fixes the loss, penalty, regularization strength, learning-rate schedule, hash dimension, and batch size; shuffles training data each epoch; evaluates validation Macro-F1; retains the best validation checkpoint; and records convergence history.


In [ ]:
# ============================================================
# 9. Reproducible hashed character n-gram SGD model
# ============================================================
def fit_hash_sgd(
    train_words, train_labels,
    val_words, val_labels,
    classes,
    seed,
    n_features=HASH_N_FEATURES,
    max_epochs=SGD_MAX_EPOCHS,
    patience=SGD_PATIENCE,
    batch_size=SGD_BATCH_SIZE,
):
    set_global_seed(seed)

    train_words = np.asarray(list(train_words), dtype=object)
    train_labels = np.asarray(list(train_labels), dtype=object)
    val_words = list(val_words)
    val_labels = np.asarray(list(val_labels), dtype=object)
    classes = np.asarray(sorted(classes), dtype=object)

    vec = HashingVectorizer(
        analyzer="char",
        ngram_range=HASH_NGRAM_RANGE,
        n_features=n_features,
        alternate_sign=False,
        norm="l2"
    )

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=train_labels
    )
    class_weight = dict(zip(classes, weights))

    clf = SGDClassifier(
        loss=SGD_LOSS,
        penalty=SGD_PENALTY,
        alpha=SGD_ALPHA,
        class_weight=class_weight,
        learning_rate=SGD_LEARNING_RATE,
        random_state=seed,
        average=False
    )

    X_val = vec.transform(val_words)
    rng = np.random.default_rng(seed)

    best_f1 = -np.inf
    best_clf = None
    no_improve = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        order = rng.permutation(len(train_words))

        for start in range(0, len(order), batch_size):
            idx = order[start:start+batch_size]
            X_batch = vec.transform(train_words[idx].tolist())
            y_batch = train_labels[idx]

            if epoch == 1 and start == 0:
                clf.partial_fit(X_batch, y_batch, classes=classes)
            else:
                clf.partial_fit(X_batch, y_batch)

        val_pred = clf.predict(X_val)
        val_f1 = f1_score(
            val_labels, val_pred,
            labels=classes, average="macro", zero_division=0
        )
        history.append({"epoch": epoch, "val_macro_f1": float(val_f1)})

        if val_f1 > best_f1 + 1e-6:
            best_f1 = val_f1
            best_clf = copy.deepcopy(clf)
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    return {
        "vectorizer": vec,
        "model": best_clf,
        "best_val_macro_f1": float(best_f1),
        "best_epoch": int(best_epoch),
        "history": pd.DataFrame(history),
    }

def predict_hash_sgd(fitted, words):
    X = fitted["vectorizer"].transform(list(words))
    return fitted["model"].predict(X)

## 10. Character-level BiLSTM

The public implementation uses:
- explicit architecture dimensions,
- class-weighted cross-entropy,
- Adam with fixed learning rate,
- validation Macro-F1 for early stopping,
- deep-copied best checkpoints,
- deterministic seeds,
- packed variable-length sequences to avoid padding artifacts,
- convergence histories.

In [ ]:
# ============================================================
# 10. Reproducible character-level BiLSTM
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

class CharDataset(Dataset):
    def __init__(self, words, labels, char2idx, label2idx, max_len=BILSTM_MAX_LEN):
        self.words = list(words)
        self.labels = list(labels)
        self.char2idx = char2idx
        self.label2idx = label2idx
        self.max_len = max_len

    def encode_word(self, word):
        chars = [
            self.char2idx.get(ch, self.char2idx["<unk>"])
            for ch in word[:self.max_len]
        ]
        length = max(1, len(chars))
        if len(chars) < self.max_len:
            chars += [self.char2idx["<pad>"]] * (self.max_len - len(chars))
        return (
            torch.tensor(chars, dtype=torch.long),
            torch.tensor(length, dtype=torch.long)
        )

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        x, length = self.encode_word(self.words[idx])
        y = torch.tensor(self.label2idx[self.labels[idx]], dtype=torch.long)
        return x, length, y

class CharBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = pack_padded_sequence(
            emb, lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        # forward final + backward final
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.fc(h)

def build_char_vocab(words):
    chars = sorted(set("".join(words)))
    char2idx = {"<pad>": 0, "<unk>": 1}
    for ch in chars:
        char2idx[ch] = len(char2idx)
    return char2idx

def fit_bilstm(train_df, val_df, label_col, seed):
    set_global_seed(seed)

    labels = sorted(train_df[label_col].unique().tolist())
    label2idx = {lab: i for i, lab in enumerate(labels)}
    idx2label = {i: lab for lab, i in label2idx.items()}
    char2idx = build_char_vocab(train_df["word"].tolist())

    train_ds = CharDataset(
        train_df["word"], train_df[label_col], char2idx, label2idx
    )
    val_ds = CharDataset(
        val_df["word"], val_df[label_col], char2idx, label2idx
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BILSTM_BATCH_SIZE,
        shuffle=True, generator=generator
    )
    val_loader = DataLoader(
        val_ds, batch_size=BILSTM_BATCH_SIZE, shuffle=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = CharBiLSTM(
        vocab_size=len(char2idx),
        embed_dim=BILSTM_EMBED_DIM,
        hidden_dim=BILSTM_HIDDEN_DIM,
        num_classes=len(labels),
        pad_idx=char2idx["<pad>"]
    ).to(device)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array(labels),
        y=np.array(train_df[label_col].tolist())
    )
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float, device=device)
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=BILSTM_LR)

    def run(loader, train=False):
        model.train() if train else model.eval()
        total_loss = 0.0
        preds, golds = [], []

        context = torch.enable_grad() if train else torch.no_grad()
        with context:
            for x, lengths, y in loader:
                x, lengths, y = x.to(device), lengths.to(device), y.to(device)

                if train:
                    optimizer.zero_grad()

                logits = model(x, lengths)
                loss = criterion(logits, y)

                if train:
                    loss.backward()
                    optimizer.step()

                total_loss += loss.item() * x.size(0)
                preds.extend(logits.argmax(1).cpu().numpy().tolist())
                golds.extend(y.cpu().numpy().tolist())

        return total_loss / len(loader.dataset), preds, golds

    best_f1 = -np.inf
    best_state = None
    best_epoch = None
    no_improve = 0
    history = []

    for epoch in range(1, BILSTM_MAX_EPOCHS + 1):
        train_loss, _, _ = run(train_loader, train=True)
        val_loss, vp, vg = run(val_loader, train=False)
        val_f1 = f1_score(vg, vp, average="macro", zero_division=0)

        history.append({
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_macro_f1": float(val_f1)
        })

        if val_f1 > best_f1 + 1e-6:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= BILSTM_PATIENCE:
            break

    model.load_state_dict(best_state)

    return {
        "model": model,
        "char2idx": char2idx,
        "label2idx": label2idx,
        "idx2label": idx2label,
        "labels": labels,
        "device": device,
        "best_val_macro_f1": float(best_f1),
        "best_epoch": int(best_epoch),
        "history": pd.DataFrame(history),
    }

def predict_bilstm(fitted, words):
    dummy_label = fitted["labels"][0]
    ds = CharDataset(
        list(words),
        [dummy_label] * len(list(words)),
        fitted["char2idx"],
        fitted["label2idx"]
    )
    loader = DataLoader(ds, batch_size=BILSTM_BATCH_SIZE, shuffle=False)

    model = fitted["model"]
    device = fitted["device"]
    model.eval()
    preds = []

    with torch.no_grad():
        for x, lengths, _ in loader:
            x, lengths = x.to(device), lengths.to(device)
            logits = model(x, lengths)
            preds.extend(logits.argmax(1).cpu().numpy().tolist())

    return [fitted["idx2label"][i] for i in preds]

## 11. Standard coarse- and fine-grained experiments

The same executable pipeline is used for both coarse- and fine-grained classification, and all reported values are computed directly from model predictions.

In [ ]:
# ============================================================
# 11. DEFINE THE STANDARD CLASSIFICATION TASK
# ============================================================

def run_standard_task(
    train_df,
    val_df,
    test_df,
    label_col,
    rule_set,
    seed=PRIMARY_SEED,
    run_bilstm=True,
    compute_ci=True,
    bootstrap_reps=BOOTSTRAP_REPS
):
    """
    Run all classification models for one task.

    Models
    ------
    1. Majority-class baseline
    2. Rule-based suffix
    3. Statistical suffix model
    4. Char N-gram linear model
    5. Character-level BiLSTM

    Parameters
    ----------
    train_df : pandas.DataFrame
        Training partition.

    val_df : pandas.DataFrame
        Validation partition.

    test_df : pandas.DataFrame
        Test partition.

    label_col : str
        Target label column.
        Examples:
        - "coarse_tag"
        - "pos_tag"

    rule_set : list
        Rule-Based suffix mappings.

    seed : int
        Random seed for stochastic models.

    run_bilstm : bool
        If True, train and evaluate the BiLSTM.

    compute_ci : bool
        If True, compute bootstrap confidence intervals.

    bootstrap_reps : int
        Number of bootstrap resamples.

    Returns
    -------
    dict containing:
        results
        predictions
        hash_fit
        bilstm_fit
        labels
        y_test
    """

    # --------------------------------------------------------
    # 1. Validate required columns
    # --------------------------------------------------------

    required_cols = {"word", label_col}

    for frame_name, frame in [
        ("train_df", train_df),
        ("val_df", val_df),
        ("test_df", test_df)
    ]:

        missing_cols = required_cols - set(frame.columns)

        if missing_cols:
            raise ValueError(
                f"{frame_name} is missing required columns: "
                f"{missing_cols}"
            )

    # --------------------------------------------------------
    # 2. Remove accidental missing values
    # --------------------------------------------------------

    train_df = train_df.dropna(
        subset=["word", label_col]
    ).copy()

    val_df = val_df.dropna(
        subset=["word", label_col]
    ).copy()

    test_df = test_df.dropna(
        subset=["word", label_col]
    ).copy()

    # --------------------------------------------------------
    # 3. Standardize data types
    # --------------------------------------------------------

    for frame in [train_df, val_df, test_df]:

        frame["word"] = (
            frame["word"]
            .astype(str)
        )

        frame[label_col] = (
            frame[label_col]
            .astype(str)
        )

    # --------------------------------------------------------
    # 4. Determine class labels
    # --------------------------------------------------------

    labels = sorted(
        train_df[label_col]
        .unique()
        .tolist()
    )

    if len(labels) < 2:
        raise ValueError(
            f"Only {len(labels)} class is present "
            f"in the training data."
        )

    # --------------------------------------------------------
    # 5. Check for unseen validation/test labels
    # --------------------------------------------------------

    unseen_val = (
        set(val_df[label_col].unique())
        - set(labels)
    )

    unseen_test = (
        set(test_df[label_col].unique())
        - set(labels)
    )

    if unseen_val:
        raise ValueError(
            "Validation set contains labels that are "
            f"absent from training: {unseen_val}"
        )

    if unseen_test:
        raise ValueError(
            "Test set contains labels that are "
            f"absent from training: {unseen_test}"
        )

    # --------------------------------------------------------
    # 6. Determine majority/default class
    # --------------------------------------------------------

    default_label = (
        train_df[label_col]
        .value_counts()
        .idxmax()
    )

    # --------------------------------------------------------
    # 7. Print task information
    # --------------------------------------------------------

    print("=" * 70)
    print(f"TASK: {label_col}")
    print("=" * 70)

    print(
        f"Training samples:   {len(train_df):,}"
    )

    print(
        f"Validation samples: {len(val_df):,}"
    )

    print(
        f"Test samples:       {len(test_df):,}"
    )

    print(
        f"Number of classes:  {len(labels)}"
    )

    print(
        f"Classes: {labels}"
    )

    print(
        f"Majority/default class: {default_label}"
    )

    print("=" * 70)

    # --------------------------------------------------------
    # 8. Initialize containers
    # --------------------------------------------------------

    predictions = {}

    hash_fit = None
    bilstm_fit = None

    # ========================================================
    # MODEL 1: Majority-class baseline
    # ========================================================

    print("\n[1/5] Majority-class baseline...")

    predictions["Majority-class baseline"] = np.asarray(
        majority_predict(
            train_df[label_col],
            len(test_df)
        ),
        dtype=str
    )

    # ========================================================
    # MODEL 2: Rule-based suffix
    # ========================================================

    print("[2/5] Rule-based suffix...")

    predictions["Rule-based suffix"] = np.asarray(
        rule_predict_many(
            test_df["word"].tolist(),
            rule_set,
            default_label
        ),
        dtype=str
    )

    # ========================================================
    # MODEL 3: Statistical suffix model
    # ========================================================

    print("[3/5] Statistical suffix model...")

    predictions["Statistical suffix"] = np.asarray(
        suffix_predict_many(
            train_df=train_df,
            test_words=test_df["word"].tolist(),
            label_col=label_col
        ),
        dtype=str
    )

    # ========================================================
    # MODEL 4: Character n-gram SGD
    # ========================================================

    print("[4/5] Character n-gram SGD...")

    hash_fit = fit_hash_sgd(
        train_words=train_df["word"].tolist(),
        train_labels=train_df[label_col].tolist(),
        val_words=val_df["word"].tolist(),
        val_labels=val_df[label_col].tolist(),
        classes=labels,
        seed=seed
    )

    predictions["Char N-gram"] = np.asarray(
        predict_hash_sgd(
            fitted=hash_fit,
            words=test_df["word"].tolist()
        ),
        dtype=str
    )

    print(
        f"    Best validation Macro-F1: "
        f"{hash_fit['best_val_macro_f1']:.4f}"
    )

    print(
        f"    Best epoch: "
        f"{hash_fit['best_epoch']}"
    )

    # ========================================================
    # MODEL 5: Character-level BiLSTM
    # ========================================================

    if run_bilstm:

        print("[5/5] Character-level BiLSTM...")

        bilstm_fit = fit_bilstm(
            train_df=train_df,
            val_df=val_df,
            label_col=label_col,
            seed=seed
        )

        predictions["BiLSTM"] = np.asarray(
            predict_bilstm(
                fitted=bilstm_fit,
                words=test_df["word"].tolist()
            ),
            dtype=str
        )

        print(
            f"    Best validation Macro-F1: "
            f"{bilstm_fit['best_val_macro_f1']:.4f}"
        )

        print(
            f"    Best epoch: "
            f"{bilstm_fit['best_epoch']}"
        )

    else:

        print(
            "[5/5] BiLSTM skipped."
        )

    # --------------------------------------------------------
    # 9. Prepare true test labels
    # --------------------------------------------------------

    y = np.asarray(
        test_df[label_col].tolist(),
        dtype=str
    )

    # --------------------------------------------------------
    # 10. Check prediction lengths
    # --------------------------------------------------------

    for model_name, pred in predictions.items():

        if len(pred) != len(y):

            raise ValueError(
                f"{model_name}: prediction length "
                f"{len(pred)} does not match "
                f"test length {len(y)}."
            )

    # --------------------------------------------------------
    # 11. Compute evaluation metrics
    # --------------------------------------------------------

    print(
        "\nCalculating performance metrics..."
    )

    rows = []

    for model_name, pred in predictions.items():

        metrics = metric_dict(
            y_true=y,
            y_pred=pred,
            labels=labels
        )

        row = {
            "Model": model_name,
            "Accuracy": metrics["accuracy"],
            "Macro_F1": metrics["macro_f1"]
        }

        # ----------------------------------------------------
        # Bootstrap confidence intervals
        # ----------------------------------------------------

        if compute_ci:

            print(
                f"    Bootstrap CI: {model_name}"
            )

            accuracy_ci = stratified_bootstrap_ci(
                y_true=y,
                y_pred=pred,
                metric="accuracy",
                labels=labels,
                n_boot=bootstrap_reps,
                seed=seed
            )

            macro_f1_ci = stratified_bootstrap_ci(
                y_true=y,
                y_pred=pred,
                metric="macro_f1",
                labels=labels,
                n_boot=bootstrap_reps,
                seed=seed
            )

            row.update({
                "Accuracy_CI_low":
                    accuracy_ci[0],

                "Accuracy_CI_high":
                    accuracy_ci[1],

                "Macro_F1_CI_low":
                    macro_f1_ci[0],

                "Macro_F1_CI_high":
                    macro_f1_ci[1]
            })

        rows.append(row)

    results = pd.DataFrame(rows)

    # --------------------------------------------------------
    # 12. Arrange model order
    # --------------------------------------------------------

    preferred_order = [
        "Majority-class baseline",
        "Rule-based suffix",
        "Statistical suffix",
        "Char N-gram",
        "BiLSTM"
    ]

    order_map = {
        model: i
        for i, model
        in enumerate(preferred_order)
    }

    results["_order"] = (
        results["Model"]
        .map(order_map)
    )

    results = (
        results
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

    print(
        f"\nCompleted task: {label_col}"
    )

    # --------------------------------------------------------
    # 13. Return all outputs
    # --------------------------------------------------------

    return {
        "results": results,
        "predictions": predictions,
        "hash_fit": hash_fit,
        "bilstm_fit": bilstm_fit,
        "labels": labels,
        "y_test": y
    }

In [ ]:
# ============================================================
# 11A. EXECUTE THE COARSE-GRAINED CLASSIFICATION TASK
# ============================================================

coarse_run = run_standard_task(
    train_df=train_c,
    val_df=val_c,
    test_df=test_c,
    label_col="coarse_tag",
    rule_set=COARSE_RULES,
    seed=PRIMARY_SEED,
    run_bilstm=True,
    compute_ci=False,
    bootstrap_reps=BOOTSTRAP_REPS
)

print(
    "\nCOARSE-GRAINED CLASSIFICATION RESULTS"
)

display(
    coarse_run["results"]
)

# Save results
coarse_run["results"].to_csv(
    OUT_DIR / "table_coarse_results.csv",
    index=False
)

# Confirm that predictions were created successfully
print(
    "\nModels evaluated:"
)

print(
    list(
        coarse_run["predictions"].keys()
    )
)

print(
    "\nNumber of coarse test observations:",
    len(coarse_run["y_test"])
)

In [ ]:
# ============================================================
# 11B. EXECUTE THE FINE-GRAINED CLASSIFICATION TASK
# ============================================================

fine_run = run_standard_task(
    train_df=train_f,
    val_df=val_f,
    test_df=test_f,
    label_col="pos_tag",
    rule_set=FINE_RULES,
    seed=PRIMARY_SEED,
    run_bilstm=True,
    compute_ci=False,
    bootstrap_reps=BOOTSTRAP_REPS
)

print(
    "\nFINE-GRAINED CLASSIFICATION RESULTS"
)

display(
    fine_run["results"]
)

# Save results
fine_run["results"].to_csv(
    OUT_DIR / "table_fine_results.csv",
    index=False
)

# Confirm successful execution
print(
    "\nModels evaluated:"
)

print(
    list(
        fine_run["predictions"].keys()
    )
)

print(
    "\nNumber of fine-grained test observations:",
    len(fine_run["y_test"])
)

print(
    "\nFine-grained classes:"
)

print(
    fine_run["labels"]
)

In [ ]:
# ============================================================
# 12. FUNCTION-CLASS RECALL WITH EXACT 95% INTERVALS
# ============================================================
# The FUNCTION class is sparse in the coarse test set. Report
# recall together with an exact Clopper-Pearson binomial interval.

function_mask = np.asarray(coarse_run["y_test"]) == "FUNCTION"
function_n = int(function_mask.sum())
function_rows = []

for model_name, pred in coarse_run["predictions"].items():
    pred = np.asarray(pred)
    correct = int(np.sum(pred[function_mask] == "FUNCTION"))
    recall = correct / function_n if function_n else np.nan
    ci_low, ci_high = clopper_pearson(correct, function_n, alpha=ALPHA)

    function_rows.append({
        "model": model_name,
        "support": function_n,
        "correct": correct,
        "recall": recall,
        "ci_lower": ci_low,
        "ci_upper": ci_high
    })

function_results = pd.DataFrame(function_rows)
display(function_results)

function_results.to_csv(
    OUT_DIR / "function_class_results.csv",
    index=False
)


In [ ]:
# ============================================================
# 13. PAIRED SIGNIFICANCE TESTS
#     Adjacent substantive model comparisons
# ============================================================

def adjacent_pair_tests(
    y_true,
    predictions,
    labels,
    n_perm=PERMUTATION_REPS,
    seed=PRIMARY_SEED
):
    """
    Perform paired randomization tests on Macro-F1 between
    adjacent substantive models.

    Comparisons:
    1. Rule-based suffix vs Statistical suffix
    2. Statistical suffix vs Char N-gram
    3. Char N-gram vs BiLSTM

    Holm adjustment is applied to the three p-values.
    """

    y_true = np.asarray(y_true, dtype=str)

    order = [
        "Rule-based suffix",
        "Statistical suffix",
        "Char N-gram",
        "BiLSTM"
    ]

    # --------------------------------------------------------
    # Check that required model predictions are available
    # --------------------------------------------------------
    missing_models = [
        model
        for model in order
        if model not in predictions
    ]

    if missing_models:
        raise ValueError(
            "Missing predictions for: "
            f"{missing_models}. "
            "Make sure Sections 11A and 11B were run with "
            "run_bilstm=True."
        )

    # --------------------------------------------------------
    # Check prediction lengths
    # --------------------------------------------------------
    for model_name in order:

        pred = np.asarray(
            predictions[model_name],
            dtype=str
        )

        if len(pred) != len(y_true):
            raise ValueError(
                f"{model_name}: prediction length "
                f"{len(pred)} does not match "
                f"y_true length {len(y_true)}."
            )

    # --------------------------------------------------------
    # Run adjacent paired tests
    # --------------------------------------------------------
    rows = []

    for model_a, model_b in zip(
        order[:-1],
        order[1:]
    ):

        delta, p_value = (
            paired_macro_f1_randomization_test(
                y_true=y_true,
                pred_a=np.asarray(
                    predictions[model_a],
                    dtype=str
                ),
                pred_b=np.asarray(
                    predictions[model_b],
                    dtype=str
                ),
                labels=labels,
                n_perm=n_perm,
                seed=seed
            )
        )

        rows.append({
            "Model_A": model_a,
            "Model_B": model_b,
            "Delta_Macro_F1_B_minus_A": delta,
            "p_raw": p_value
        })

    out = pd.DataFrame(rows)

    # --------------------------------------------------------
    # Holm correction
    # --------------------------------------------------------
    out["p_Holm"] = holm_adjust(
        out["p_raw"].to_numpy()
    )

    # Optional significance flag
    out["Significant_after_Holm"] = (
        out["p_Holm"] < 0.05
    )

    return out

In [ ]:
# ============================================================
# 13A. Check required experiment outputs
# ============================================================

if "coarse_run" not in globals():
    raise RuntimeError(
        "coarse_run is not available. "
        "Run Section 11A successfully before Section 13."
    )

if "fine_run" not in globals():
    raise RuntimeError(
        "fine_run is not available. "
        "Run Section 11B successfully before Section 13."
    )

In [ ]:
# ============================================================
# 13B. Coarse-grained paired significance tests
# ============================================================

coarse_pair_tests = adjacent_pair_tests(
    y_true=coarse_run["y_test"],
    predictions=coarse_run["predictions"],
    labels=coarse_run["labels"],
    n_perm=PERMUTATION_REPS,
    seed=PRIMARY_SEED
)

print(
    "\nCOARSE-GRAINED PAIRED SIGNIFICANCE TESTS"
)

display(
    coarse_pair_tests
)

coarse_pair_tests.to_csv(
    OUT_DIR / "paired_tests_coarse.csv",
    index=False
)

In [ ]:
# ============================================================
# 13C. Fine-grained paired significance tests
# ============================================================

fine_pair_tests = adjacent_pair_tests(
    y_true=fine_run["y_test"],
    predictions=fine_run["predictions"],
    labels=fine_run["labels"],
    n_perm=PERMUTATION_REPS,
    seed=PRIMARY_SEED
)

print(
    "\nFINE-GRAINED PAIRED SIGNIFICANCE TESTS"
)

display(
    fine_pair_tests
)

fine_pair_tests.to_csv(
    OUT_DIR / "paired_tests_fine.csv",
    index=False
)

## 15. One-family-at-a-time suffix hold-out evaluation

For each primary suffix family:
1. all words assigned to that family form the test subset;
2. all remaining words form the training pool;
3. the **rule-based suffix (masked)** baseline removes the corresponding held-out suffix rule;
4. the full-rule condition is retained only as a diagnostic reference and is excluded from the primary comparison;
5. Macro-F1 is calculated only over classes with ground-truth support in the held-out test subset; and
6. class support and per-class F1 are exported.


In [ ]:
# ============================================================
# 15. SUFFIX FAMILY DEFINITIONS
# ============================================================

# ------------------------------------------------------------
# 15.1 Common suffix inventory
# ------------------------------------------------------------
# Used to assign each word to one mutually exclusive suffix
# family. Longest matching suffix takes precedence.
#
# This broader inventory is used for descriptive analysis.
# The primary experimental hold-out families are defined
# separately below.
# ------------------------------------------------------------

COMMON_SUFFIXES = [
    # Nominal tendencies
    "tion",
    "sion",
    "ment",
    "ness",
    "ity",
    "ism",
    "ist",

    # Adjectival tendencies
    "ous",
    "ive",
    "able",
    "ible",
    "ful",
    "less",
    "ish",
    "al",
    "ic",

    # Adverbial tendency
    "ly",

    # Verbal / inflectional tendencies
    "ing",
    "ed",
    "en",

    # Comparative / superlative
    "er",
    "est"
]


# ------------------------------------------------------------
# 15.2 Primary suffix families used in hold-out experiments
# ------------------------------------------------------------
# These are evaluated ONE AT A TIME.
# ------------------------------------------------------------

PRIMARY_HOLDOUT_SUFFIXES = [
    "tion",
    "ness",
    "ly",
    "ing"
]


# ------------------------------------------------------------
# 15.3 Assign each word to a suffix family
# ------------------------------------------------------------

def assign_suffix_family(
    word,
    suffixes=COMMON_SUFFIXES
):
    """
    Assign a word to its longest matching suffix family.

    If no listed suffix matches, return 'OTHER'.

    Longest-match precedence prevents shorter suffixes
    from overriding longer suffixes.
    """

    word = str(word).lower()

    for suffix in sorted(
        suffixes,
        key=len,
        reverse=True
    ):

        if word.endswith(suffix):
            return suffix

    return "OTHER"


# ------------------------------------------------------------
# 15.4 Create suffix-family dataset
# ------------------------------------------------------------

holdout_df = df_coarse.copy()

holdout_df["suffix_family"] = (
    holdout_df["word"]
    .map(assign_suffix_family)
)


# ------------------------------------------------------------
# 15.5 Count observations by suffix family
# ------------------------------------------------------------

family_counts = (
    holdout_df["suffix_family"]
    .value_counts()
    .rename_axis("suffix_family")
    .reset_index(name="count")
)


# ------------------------------------------------------------
# 15.6 Calculate percentage of dataset
# ------------------------------------------------------------

family_counts["percent"] = (
    100
    * family_counts["count"]
    / len(holdout_df)
)


# ------------------------------------------------------------
# 15.7 Identify primary experimental hold-out families
# ------------------------------------------------------------

family_counts["primary_holdout"] = (
    family_counts["suffix_family"]
    .isin(PRIMARY_HOLDOUT_SUFFIXES)
)


# ------------------------------------------------------------
# 15.8 Display suffix-family distribution
# ------------------------------------------------------------

print("=" * 70)
print("SUFFIX FAMILY DISTRIBUTION")
print("=" * 70)

display(
    family_counts
)


# ------------------------------------------------------------
# 15.9 Display primary hold-out families separately
# ------------------------------------------------------------

primary_family_counts = (
    family_counts[
        family_counts["primary_holdout"]
    ]
    .copy()
)

print(
    "\nPRIMARY SUFFIX HOLD-OUT FAMILIES"
)

display(
    primary_family_counts
)


# ------------------------------------------------------------
# 15.10 Check that all primary hold-out families are present
# ------------------------------------------------------------

observed_families = set(
    holdout_df["suffix_family"]
    .unique()
)

missing_holdouts = [
    suffix
    for suffix in PRIMARY_HOLDOUT_SUFFIXES
    if suffix not in observed_families
]

if missing_holdouts:

    raise ValueError(
        "The following primary hold-out suffixes "
        f"were not found in the dataset: "
        f"{missing_holdouts}"
    )

else:

    print(
        "\nAll primary hold-out suffix families "
        "are present in the dataset."
    )


# ------------------------------------------------------------
# 15.11 Check mutually exclusive family assignment
# ------------------------------------------------------------

assert (
    holdout_df["suffix_family"]
    .notna()
    .all()
)

assert (
    len(holdout_df)
    == family_counts["count"].sum()
)

print(
    "All words were assigned to exactly one "
    "suffix family."
)


# ------------------------------------------------------------
# 15.12 Save outputs
# ------------------------------------------------------------

family_counts.to_csv(
    OUT_DIR / "suffix_family_counts.csv",
    index=False
)

primary_family_counts.to_csv(
    OUT_DIR / "primary_holdout_family_counts.csv",
    index=False
)

In [ ]:
PRIMARY_HOLDOUT_SUFFIXES = [
    "tion",
    "ness",
    "ly",
    "ing"
]

In [ ]:
# ============================================================
# 16. PER-FAMILY SUFFIX HOLD-OUT EVALUATION
# ============================================================

# ------------------------------------------------------------
# 16.1 Macro-F1 restricted to classes present in the
#      held-out test family
# ------------------------------------------------------------

def present_class_macro_f1(
    y_true,
    y_pred
):
    """
    Compute Macro-F1 only over classes that are actually
    represented in the ground-truth held-out test subset.

    Classes with zero ground-truth support are excluded.
    """

    y_true = np.asarray(
        y_true,
        dtype=str
    )

    y_pred = np.asarray(
        y_pred,
        dtype=str
    )

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred must have the same length."
        )

    present_classes = sorted(
        pd.unique(y_true).tolist()
    )

    score = f1_score(
        y_true,
        y_pred,
        labels=present_classes,
        average="macro",
        zero_division=0
    )

    return (
        float(score),
        present_classes
    )


# ------------------------------------------------------------
# 16.2 Per-class F1 for each held-out suffix family
# ------------------------------------------------------------

def per_class_f1_rows(
    y_true,
    y_pred,
    family,
    model
):
    """
    Return per-class F1 and support for classes represented
    in one held-out suffix-family test subset.
    """

    y_true = np.asarray(
        y_true,
        dtype=str
    )

    y_pred = np.asarray(
        y_pred,
        dtype=str
    )

    if len(y_true) != len(y_pred):
        raise ValueError(
            f"{model}: prediction length does not match "
            "the test-label length."
        )

    present_classes = sorted(
        pd.unique(y_true).tolist()
    )

    rows = []

    for label in present_classes:

        class_f1 = f1_score(
            y_true,
            y_pred,
            labels=[label],
            average="macro",
            zero_division=0
        )

        support = int(
            np.sum(
                y_true == label
            )
        )

        rows.append({
            "held_out_suffix": family,
            "model": model,
            "class": label,
            "support": support,
            "f1": float(class_f1)
        })

    return rows


# ============================================================
# 16.3 Evaluate one suffix family
# ============================================================

def evaluate_suffix_family(
    family,
    seed=PRIMARY_SEED,
    run_bilstm_model=True
):
    """
    Evaluate morphological generalization for one held-out
    suffix family.

    The selected family is completely excluded from model
    training and validation.

    Models:
    1. Majority
    2. Rule-Full Reference
    3. Rule-Masked
    4. Statistical Suffix
    5. Character n-gram SGD
    6. Character-level BiLSTM

    Rule-Full Reference retains the held-out suffix rule and
    is treated only as an external linguistic-reference
    condition.

    Rule-Masked removes the corresponding suffix rule and is
    the fair rule-based generalization baseline.
    """

    # --------------------------------------------------------
    # 1. Validate suffix family
    # --------------------------------------------------------

    if family not in PRIMARY_HOLDOUT_SUFFIXES:
        raise ValueError(
            f"'{family}' is not one of the primary "
            f"hold-out suffixes: "
            f"{PRIMARY_HOLDOUT_SUFFIXES}"
        )

    # --------------------------------------------------------
    # 2. Separate held-out family from remaining data
    # --------------------------------------------------------

    non_holdout_pool = (
        holdout_df[
            holdout_df["suffix_family"] != family
        ]
        .copy()
    )

    test = (
        holdout_df[
            holdout_df["suffix_family"] == family
        ]
        .copy()
    )

    if len(test) == 0:
        raise ValueError(
            f"No test examples were found for "
            f"suffix family '{family}'."
        )

    if len(non_holdout_pool) == 0:
        raise ValueError(
            f"No training examples remain after "
            f"holding out '{family}'."
        )

    # --------------------------------------------------------
    # 3. Verify lexical separation
    # --------------------------------------------------------

    train_words_all = set(
        non_holdout_pool["word"]
        .astype(str)
    )

    test_words = set(
        test["word"]
        .astype(str)
    )

    if not train_words_all.isdisjoint(test_words):
        raise ValueError(
            f"Lexical overlap detected for held-out "
            f"family '{family}'."
        )

    # --------------------------------------------------------
    # 4. Create leakage-free training/validation split
    # --------------------------------------------------------
    # 85% of the non-held-out pool is used for training.
    # 15% is used for validation.
    #
    # The held-out suffix family remains completely untouched
    # and serves exclusively as the test set.
    # --------------------------------------------------------

    tr, va = train_test_split(
        non_holdout_pool,
        test_size=0.15,
        stratify=non_holdout_pool["coarse_tag"],
        random_state=SPLIT_SEED
    )

    tr = tr.copy()
    va = va.copy()

    # --------------------------------------------------------
    # 5. Explicit leakage checks
    # --------------------------------------------------------

    assert not (
        tr["suffix_family"] == family
    ).any(), (
        f"Held-out suffix '{family}' leaked "
        "into training data."
    )

    assert not (
        va["suffix_family"] == family
    ).any(), (
        f"Held-out suffix '{family}' leaked "
        "into validation data."
    )

    assert (
        test["suffix_family"] == family
    ).all(), (
        f"Test set contains observations outside "
        f"the '{family}' family."
    )

    assert set(
        tr["word"]
    ).isdisjoint(
        set(test["word"])
    )

    assert set(
        va["word"]
    ).isdisjoint(
        set(test["word"])
    )

    # --------------------------------------------------------
    # 6. Prepare labels
    # --------------------------------------------------------

    labels_train = sorted(
        tr["coarse_tag"]
        .unique()
        .tolist()
    )

    y_test = np.asarray(
        test["coarse_tag"].tolist(),
        dtype=str
    )

    present_classes = sorted(
        pd.unique(y_test).tolist()
    )

    # Check that all test classes are represented in training
    unseen_test_classes = (
        set(present_classes)
        - set(labels_train)
    )

    if unseen_test_classes:
        raise ValueError(
            f"Held-out family '{family}' contains "
            f"classes absent from training: "
            f"{unseen_test_classes}"
        )

    # --------------------------------------------------------
    # 7. Default / majority class
    # --------------------------------------------------------

    default_label = (
        tr["coarse_tag"]
        .value_counts()
        .idxmax()
    )

    # --------------------------------------------------------
    # 8. Display family information
    # --------------------------------------------------------

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"HELD-OUT SUFFIX FAMILY: -{family}"
    )

    print(
        "=" * 70
    )

    print(
        f"Training n:   {len(tr):,}"
    )

    print(
        f"Validation n: {len(va):,}"
    )

    print(
        f"Test n:       {len(test):,}"
    )

    print(
        f"Test classes: {present_classes}"
    )

    print(
        "Test class support:"
    )

    print(
        test["coarse_tag"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    # --------------------------------------------------------
    # 9. Predictions container
    # --------------------------------------------------------

    predictions = {}

    # ========================================================
    # MODEL 1: Majority-class baseline
    # ========================================================

    print(
        "\n[1/6] Majority-class baseline..."
    )

    predictions["Majority-class baseline"] = np.asarray(
        majority_predict(
            tr["coarse_tag"],
            len(test)
        ),
        dtype=str
    )

    # ========================================================
    # MODEL 2: Full rule reference
    # ========================================================

    print(
        "[2/6] Full rule reference..."
    )

    predictions["Rule-based suffix (full reference)"] = np.asarray(
        rule_predict_many(
            test["word"].tolist(),
            COARSE_RULES,
            default_label
        ),
        dtype=str
    )

    # ========================================================
    # MODEL 3: Masked rule baseline
    # ========================================================

    print(
        "[3/6] Masked rule baseline..."
    )

    predictions["Rule-based suffix (masked)"] = np.asarray(
        rule_predict_many(
            test["word"].tolist(),
            COARSE_RULES,
            default_label,
            masked_suffixes={family}
        ),
        dtype=str
    )

    # ========================================================
    # MODEL 4: Statistical suffix model
    # ========================================================

    print(
        "[4/6] Statistical suffix model..."
    )

    predictions["Statistical suffix"] = np.asarray(
        suffix_predict_many(
            train_df=tr,
            test_words=test["word"].tolist(),
            label_col="coarse_tag"
        ),
        dtype=str
    )

    # ========================================================
    # MODEL 5: Character n-gram SGD
    # ========================================================

    print(
        "[5/6] Character n-gram SGD..."
    )

    hash_fit = fit_hash_sgd(
        train_words=tr["word"].tolist(),
        train_labels=tr["coarse_tag"].tolist(),
        val_words=va["word"].tolist(),
        val_labels=va["coarse_tag"].tolist(),
        classes=labels_train,
        seed=seed
    )

    predictions["Char N-gram"] = np.asarray(
        predict_hash_sgd(
            hash_fit,
            test["word"].tolist()
        ),
        dtype=str
    )

    # Save convergence history
    hash_fit["history"].to_csv(
        OUT_DIR
        / f"sgd_holdout_{family}_seed{seed}.csv",
        index=False
    )

    # ========================================================
    # MODEL 6: Character-level BiLSTM
    # ========================================================

    bilstm_fit = None

    if run_bilstm_model:

        print(
            "[6/6] Character-level BiLSTM..."
        )

        bilstm_fit = fit_bilstm(
            train_df=tr,
            val_df=va,
            label_col="coarse_tag",
            seed=seed
        )

        predictions["BiLSTM"] = np.asarray(
            predict_bilstm(
                bilstm_fit,
                test["word"].tolist()
            ),
            dtype=str
        )

        # Save convergence history
        bilstm_fit["history"].to_csv(
            OUT_DIR
            / f"bilstm_holdout_{family}_seed{seed}.csv",
            index=False
        )

    else:

        print(
            "[6/6] BiLSTM skipped."
        )

    # --------------------------------------------------------
    # 10. Validate prediction lengths
    # --------------------------------------------------------

    for model_name, pred in predictions.items():

        if len(pred) != len(y_test):

            raise ValueError(
                f"{model_name}: prediction length "
                f"{len(pred)} does not match "
                f"test length {len(y_test)}."
            )

    # --------------------------------------------------------
    # 11. Class-support information
    # --------------------------------------------------------

    support_dict = (
        test["coarse_tag"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    # --------------------------------------------------------
    # 12. Calculate family-level and class-level metrics
    # --------------------------------------------------------

    summary_rows = []
    class_rows = []

    for model_name, pred in predictions.items():

        macro_present, classes_used = (
            present_class_macro_f1(
                y_test,
                pred
            )
        )

        summary_rows.append({
            "held_out_suffix":
                family,

            "train_n":
                len(tr),

            "validation_n":
                len(va),

            "test_n":
                len(test),

            "classes_present":
                ",".join(classes_used),

            "n_classes_present":
                len(classes_used),

            "class_support":
                json.dumps(
                    support_dict,
                    sort_keys=True
                ),

            "model":
                model_name,

            "macro_f1_present_classes":
                macro_present,

            "seed":
                seed
        })

        class_rows.extend(
            per_class_f1_rows(
                y_true=y_test,
                y_pred=pred,
                family=family,
                model=model_name
            )
        )

    # --------------------------------------------------------
    # 13. Convert results to DataFrames
    # --------------------------------------------------------

    summary_df = pd.DataFrame(
        summary_rows
    )

    class_df = pd.DataFrame(
        class_rows
    )

    return (
        summary_df,
        class_df
    )

In [ ]:
# ============================================================
# 16.4 RUN PRIMARY SUFFIX HOLD-OUT EXPERIMENTS
# ============================================================

all_family_summary = []
all_family_class = []

for family in PRIMARY_HOLDOUT_SUFFIXES:

    print(
        "\nRunning held-out family:",
        family
    )

    summary_df, class_df = (
        evaluate_suffix_family(
            family=family,
            seed=PRIMARY_SEED,
            run_bilstm_model=True
        )
    )

    all_family_summary.append(
        summary_df
    )

    all_family_class.append(
        class_df
    )


# ------------------------------------------------------------
# Combine all families
# ------------------------------------------------------------

suffix_summary = pd.concat(
    all_family_summary,
    ignore_index=True
)

suffix_classwise = pd.concat(
    all_family_class,
    ignore_index=True
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "\nSUFFIX HOLD-OUT SUMMARY"
)

display(
    suffix_summary
)

print(
    "\nSUFFIX HOLD-OUT CLASS-WISE RESULTS"
)

display(
    suffix_classwise
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

suffix_summary.to_csv(
    OUT_DIR
    / "suffix_holdout_per_family.csv",
    index=False
)

suffix_classwise.to_csv(
    OUT_DIR
    / "suffix_holdout_classwise.csv",
    index=False
)

### Interpreting hold-out Macro-F1

The manuscript should state explicitly:

> For each suffix-family hold-out experiment, Macro-F1 was computed only over POS classes represented in the ground-truth held-out subset. Classes with zero ground-truth support were excluded from that family's Macro-F1 average. Per-class support and F1 scores are reported alongside the aggregate score.

This prevents zero-support classes from dominating or distorting a suffix-specific score.

In [ ]:
# ============================================================
# 17. CONVERGENCE PLOTS
# ============================================================

def plot_training_history(
    history,
    model_name,
    file_stem
):
    """
    Plot training/validation loss and validation Macro-F1
    across epochs.

    The figure is saved as a 300-dpi PNG in OUT_DIR.
    """

    if history is None or len(history) == 0:
        raise ValueError(
            f"No training history available for {model_name}."
        )

    required = {"epoch"}

    if not required.issubset(history.columns):
        raise ValueError(
            f"Training history for {model_name} "
            "does not contain an 'epoch' column."
        )

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, ax1 = plt.subplots(
        figsize=(6, 4)
    )

    handles = []
    labels = []

    # --------------------------------------------------------
    # Training loss
    # --------------------------------------------------------

    if "train_loss" in history.columns:

        line, = ax1.plot(
            history["epoch"],
            history["train_loss"],
            label="Training loss"
        )

        handles.append(line)
        labels.append("Training loss")

    # --------------------------------------------------------
    # Validation loss
    # --------------------------------------------------------

    if "val_loss" in history.columns:

        line, = ax1.plot(
            history["epoch"],
            history["val_loss"],
            label="Validation loss"
        )

        handles.append(line)
        labels.append("Validation loss")

    ax1.set_xlabel(
        "Epoch"
    )

    ax1.set_ylabel(
        "Loss"
    )

    # --------------------------------------------------------
    # Validation Macro-F1
    # --------------------------------------------------------

    if "val_macro_f1" in history.columns:

        ax2 = ax1.twinx()

        line, = ax2.plot(
            history["epoch"],
            history["val_macro_f1"],
            linestyle="--",
            label="Validation Macro-F1"
        )

        handles.append(line)
        labels.append(
            "Validation Macro-F1"
        )

        ax2.set_ylabel(
            "Macro-F1"
        )

        ax2.set_ylim(
            0,
            1
        )

    # --------------------------------------------------------
    # Figure title and legend
    # --------------------------------------------------------

    ax1.set_title(
        model_name
    )

    if handles:
        ax1.legend(
            handles,
            labels,
            loc="best"
        )

    fig.tight_layout()

    # --------------------------------------------------------
    # Save figure
    # --------------------------------------------------------

    output_file = (
        OUT_DIR / f"{file_stem}.png"
    )

    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(
        f"Saved: {output_file}"
    )

In [ ]:
# ============================================================
# 17A. COARSE-GRAINED CONVERGENCE
# ============================================================

if "coarse_run" not in globals():
    raise RuntimeError(
        "coarse_run is not available. "
        "Run Section 11A before Section 17."
    )


# Character n-gram SGD
if coarse_run["hash_fit"] is not None:

    plot_training_history(
        history=coarse_run["hash_fit"]["history"],
        model_name=(
            "Character n-gram SGD "
            "Convergence — Coarse-Grained Task"
        ),
        file_stem="sgd_convergence_coarse"
    )


# Character-level BiLSTM
if coarse_run["bilstm_fit"] is not None:

    plot_training_history(
        history=coarse_run["bilstm_fit"]["history"],
        model_name=(
            "Character-Level BiLSTM "
            "Convergence — Coarse-Grained Task"
        ),
        file_stem="bilstm_convergence_coarse"
    )

In [ ]:
# ============================================================
# 17B. FINE-GRAINED CONVERGENCE
# ============================================================

if "fine_run" not in globals():
    raise RuntimeError(
        "fine_run is not available. "
        "Run Section 11B before Section 17."
    )


# Character n-gram SGD
if fine_run["hash_fit"] is not None:

    plot_training_history(
        history=fine_run["hash_fit"]["history"],
        model_name=(
            "Character n-gram SGD "
            "Convergence — Fine-Grained Task"
        ),
        file_stem="sgd_convergence_fine"
    )


# Character-level BiLSTM
if fine_run["bilstm_fit"] is not None:

    plot_training_history(
        history=fine_run["bilstm_fit"]["history"],
        model_name=(
            "Character-Level BiLSTM "
            "Convergence — Fine-Grained Task"
        ),
        file_stem="bilstm_convergence_fine"
    )

In [ ]:
# ============================================================
# 19. SAVE FULL EXPERIMENTAL CONFIGURATION
# ============================================================

import sys
import platform
import sklearn

# PyTorch version, if available
try:
    import torch
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()

    if cuda_available:
        device_name = torch.cuda.get_device_name(0)
    else:
        device_name = "CPU"

except ImportError:
    torch_version = None
    cuda_available = False
    device_name = "PyTorch not installed"


# ------------------------------------------------------------
# 19.1 Experimental configuration
# ------------------------------------------------------------

config = {

    # ========================================================
    # DATA
    # ========================================================

    "data": {
        "data_file": str(DATA_PATH),
        "word_column": "word",
        "label_column": "pos_tag",
        "normalization": "strip and lowercase",
        "alphabetic_only": True,
        "duplicate_handling":
            "duplicate word-label pairs removed after normalization",
        "fine_grained_minimum_count": MIN_COUNT_FINE,
    },


    # ========================================================
    # STANDARD TRAIN / VALIDATION / TEST SPLIT
    # ========================================================

    "standard_split": {
        "train_fraction": 0.70,
        "validation_fraction": 0.15,
        "test_fraction": 0.15,
        "stratified": True,
        "random_seed": SPLIT_SEED,
    },


    # ========================================================
    # RANDOM SEEDS
    # ========================================================

    "random_seeds": {
        "primary_seed": PRIMARY_SEED,
    },


    # ========================================================
    # TASKS
    # ========================================================

    "tasks": {

        "coarse": {
            "label_column": "coarse_tag",
            "classes": [
                "NOUN",
                "VERB",
                "ADJ",
                "ADV",
                "FUNCTION"
            ]
        },

        "fine": {
            "label_column": "pos_tag",
            "minimum_class_count": MIN_COUNT_FINE,
        }
    },


    # ========================================================
    # MAJORITY BASELINE
    # ========================================================

    "majority_baseline": {
        "prediction_rule":
            "most frequent label in the training partition"
    },


    # ========================================================
    # RULE-BASED SUFFIX MODEL
    # ========================================================

    "rule_based_model": {
        "matching_strategy": "longest matching suffix",
        "fallback":
            "majority class of the corresponding training partition",
        "coarse_rules": COARSE_RULES,
        "fine_rules": FINE_RULES,
    },


    # ========================================================
    # LEARNED / STATISTICAL SUFFIX MODEL
    # ========================================================

    "learned_suffix_model": {
        "minimum_suffix_length": 1,
        "maximum_suffix_length": 5,
        "matching_strategy":
            "longest observed matching suffix",
        "label_estimation":
            "empirical suffix-label frequency",
        "smoothing": "none",
        "unseen_suffix_fallback":
            "majority class of training partition",
    },


    # ========================================================
    # CHARACTER N-GRAM SGD
    # ========================================================

    "character_ngram_sgd": {
        "vectorizer": "HashingVectorizer",
        "analyzer": "char",
        "n_features": HASH_N_FEATURES,
        "ngram_range": list(HASH_NGRAM_RANGE),
        "alternate_sign": False,
        "norm": "l2",

        "classifier": "SGDClassifier",
        "loss": SGD_LOSS,
        "penalty": SGD_PENALTY,
        "alpha": SGD_ALPHA,
        "learning_rate": SGD_LEARNING_RATE,

        "max_epochs": SGD_MAX_EPOCHS,
        "early_stopping_patience": SGD_PATIENCE,
        "batch_size": SGD_BATCH_SIZE,

        "selection_metric":
            "validation Macro-F1",
    },


    # ========================================================
    # CHARACTER-LEVEL BiLSTM
    # ========================================================

    "bilstm": {
        "input_unit": "character",
        "max_word_length": BILSTM_MAX_LEN,
        "embedding_dim": BILSTM_EMBED_DIM,
        "hidden_dim_per_direction": BILSTM_HIDDEN_DIM,
        "bidirectional": True,

        "sequence_representation":
            "final forward and backward hidden states",

        "batch_size": BILSTM_BATCH_SIZE,

        "loss":
            "class-weighted cross-entropy",

        "optimizer": "Adam",
        "learning_rate": BILSTM_LR,
        "learning_rate_schedule": "fixed",

        "max_epochs": BILSTM_MAX_EPOCHS,
        "early_stopping_patience": BILSTM_PATIENCE,

        "selection_metric":
            "validation Macro-F1",

        "padding":
            "packed variable-length sequences",

        "checkpointing":
            "best validation Macro-F1 state copied using deepcopy",
    },


    # ========================================================
    # EVALUATION METRICS
    # ========================================================

    "evaluation": {
        "primary_metric": "Macro-F1",

        "additional_metrics": [
            "Accuracy"
        ],

        "standard_macro_f1":
            "computed over task label set",

        "holdout_macro_f1":
            "computed only over classes present in the ground-truth "
            "held-out suffix-family test subset",

        "zero_division": 0,
    },


    # ========================================================
    # UNCERTAINTY
    # ========================================================

    "uncertainty": {
        "bootstrap_repetitions": BOOTSTRAP_REPS,
        "confidence_level": 0.95,
        "function_recall_interval":
            "Clopper-Pearson exact interval",
    },


    # ========================================================
    # PAIRED SIGNIFICANCE TESTING
    # ========================================================

    "paired_significance_tests": {
        "method":
            "paired randomization test on Macro-F1",

        "repetitions": PERMUTATION_REPS,

        "comparisons": [
            [
                "Rule-based suffix",
                "Statistical Suffix"
            ],
            [
                "Statistical suffix",
                "Character N-gram"
            ],
            [
                "Char N-gram",
                "BiLSTM"
            ]
        ],

        "multiple_testing_correction":
            "Holm",

        "alpha": 0.05,
    },


    # ========================================================
    # MULTI-SEED EVALUATION
    # ========================================================

    "multi_seed_evaluation": {
        "models": [
            "Char N-gram",
            "BiLSTM"
        ],

        "seeds": list(MODEL_SEEDS),

        "reported_statistics": [
            "mean",
            "standard deviation"
        ]
    },


    # ========================================================
    # SUFFIX HOLD-OUT EXPERIMENT
    # ========================================================

    "suffix_holdout": {

        "common_suffix_inventory":
            list(COMMON_SUFFIXES),

        "primary_holdout_suffixes":
            list(PRIMARY_HOLDOUT_SUFFIXES),

        "family_assignment":
            "longest matching suffix",

        "unmatched_family":
            "OTHER",

        "evaluation_strategy":
            "one suffix family held out at a time",

        "training_pool":
            "all observations not assigned to the held-out family",

        "training_fraction_of_non_holdout_pool":
            0.85,

        "validation_fraction_of_non_holdout_pool":
            0.15,

        "validation_split_seed":
            SPLIT_SEED,

        "held_out_test_set":
            "all observations assigned to the selected suffix family",

        "rule_full_reference":
            "retains the held-out suffix rule and is treated only "
            "as an external linguistic-reference condition",

        "rule_masked":
            "removes the corresponding held-out suffix rule",

        "macro_f1_definition":
            "classes with zero ground-truth support in the held-out "
            "test subset are excluded",
    },


    # ========================================================
    # SOFTWARE / COMPUTING ENVIRONMENT
    # ========================================================

    "software_environment": {
        "python_version": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "scikit_learn_version": sklearn.__version__,
        "pytorch_version": torch_version,
        "cuda_available": cuda_available,
        "device": device_name,
    }
}


# ------------------------------------------------------------
# 19.2 Save configuration
# ------------------------------------------------------------

configuration_path = (
    OUT_DIR / "experimental_configuration.json"
)

with open(
    configuration_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 19.3 Display saved configuration
# ------------------------------------------------------------

print(
    json.dumps(
        config,
        indent=2,
        ensure_ascii=False
    )
)

print(
    f"\nExperimental configuration saved to: "
    f"{configuration_path}"
)

In [ ]:
# ============================================================
# 20A. CREATE MASTER STANDARD RESULTS TABLE
# ============================================================

def bootstrap_metric_ci(
    y_true,
    y_pred,
    metric_name,
    labels,
    n_bootstrap=BOOTSTRAP_REPS,
    seed=PRIMARY_SEED,
    confidence=0.95
):
    """
    Calculate a percentile bootstrap confidence interval
    for Accuracy or Macro-F1.
    """

    y_true = np.asarray(y_true, dtype=str)
    y_pred = np.asarray(y_pred, dtype=str)

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred must have the same length."
        )

    rng = np.random.default_rng(seed)

    n = len(y_true)
    scores = []

    for _ in range(n_bootstrap):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        yt = y_true[idx]
        yp = y_pred[idx]

        if metric_name == "accuracy":

            score = accuracy_score(
                yt,
                yp
            )

        elif metric_name == "macro_f1":

            score = f1_score(
                yt,
                yp,
                labels=labels,
                average="macro",
                zero_division=0
            )

        else:

            raise ValueError(
                f"Unknown metric: {metric_name}"
            )

        scores.append(
            float(score)
        )

    alpha = 1.0 - confidence

    lower = np.quantile(
        scores,
        alpha / 2
    )

    upper = np.quantile(
        scores,
        1 - alpha / 2
    )

    return (
        float(lower),
        float(upper)
    )

In [ ]:
# ============================================================
# 20B. EXTRACT STANDARD TASK RESULTS
# ============================================================

def build_standard_results(
    run_output,
    task_name,
    seed=PRIMARY_SEED,
    bootstrap_reps=BOOTSTRAP_REPS
):
    """
    Create model-level performance rows for one standard task.

    Metrics:
    - Accuracy
    - Macro-F1

    Each metric includes a 95% bootstrap confidence interval.
    """

    required_keys = {
        "predictions",
        "labels",
        "y_test"
    }

    missing = (
        required_keys
        - set(run_output.keys())
    )

    if missing:
        raise ValueError(
            f"{task_name}: missing run-output keys: {missing}"
        )

    y_true = np.asarray(
        run_output["y_test"],
        dtype=str
    )

    labels = list(
        run_output["labels"]
    )

    rows = []

    for model_name, prediction in (
        run_output["predictions"].items()
    ):

        print(
            f"Calculating results: "
            f"{task_name} / {model_name}"
        )

        y_pred = np.asarray(
            prediction,
            dtype=str
        )

        if len(y_pred) != len(y_true):

            raise ValueError(
                f"{task_name} / {model_name}: "
                "prediction length mismatch."
            )

        # ----------------------------------------------------
        # Point estimates
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_true,
            y_pred
        )

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        )

        # ----------------------------------------------------
        # Bootstrap confidence intervals
        # ----------------------------------------------------

        acc_low, acc_high = bootstrap_metric_ci(
            y_true=y_true,
            y_pred=y_pred,
            metric_name="accuracy",
            labels=labels,
            n_bootstrap=bootstrap_reps,
            seed=seed
        )

        macro_low, macro_high = bootstrap_metric_ci(
            y_true=y_true,
            y_pred=y_pred,
            metric_name="macro_f1",
            labels=labels,
            n_bootstrap=bootstrap_reps,
            seed=seed
        )

        # ----------------------------------------------------
        # Save row
        # ----------------------------------------------------

        rows.append({

            "task":
                task_name,

            "model":
                model_name,

            "test_n":
                len(y_true),

            "n_classes":
                len(labels),

            "classes":
                ",".join(labels),

            "seed":
                seed,

            "accuracy":
                float(accuracy),

            "accuracy_ci_lower":
                acc_low,

            "accuracy_ci_upper":
                acc_high,

            "macro_f1":
                float(macro_f1),

            "macro_f1_ci_lower":
                macro_low,

            "macro_f1_ci_upper":
                macro_high,

            "bootstrap_reps":
                bootstrap_reps,

            "ci_level":
                0.95
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 20C. BUILD MASTER RESULTS TABLE
# ============================================================

if "coarse_run" not in globals():
    raise RuntimeError(
        "coarse_run is not available. "
        "Run Section 11A first."
    )

if "fine_run" not in globals():
    raise RuntimeError(
        "fine_run is not available. "
        "Run Section 11B first."
    )


# ------------------------------------------------------------
# Coarse-grained results
# ------------------------------------------------------------

coarse_standard_results = build_standard_results(
    run_output=coarse_run,
    task_name="coarse",
    seed=PRIMARY_SEED,
    bootstrap_reps=BOOTSTRAP_REPS
)


# ------------------------------------------------------------
# Fine-grained results
# ------------------------------------------------------------

fine_standard_results = build_standard_results(
    run_output=fine_run,
    task_name="fine",
    seed=PRIMARY_SEED,
    bootstrap_reps=BOOTSTRAP_REPS
)


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

standard_results = pd.concat(
    [
        coarse_standard_results,
        fine_standard_results
    ],
    ignore_index=True
)

In [ ]:
# ============================================================
# 20D. STANDARDIZE MODEL ORDER
# ============================================================

MODEL_ORDER = [
    "Majority-class baseline",
    "Rule-based suffix",
    "Statistical suffix",
    "Char N-gram",
    "BiLSTM"
]


standard_results["model"] = pd.Categorical(
    standard_results["model"],
    categories=MODEL_ORDER,
    ordered=True
)


standard_results = (
    standard_results
    .sort_values(
        ["task", "model"]
    )
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# 20E. SAVE MASTER RESULTS
# ============================================================

standard_results.to_csv(
    OUT_DIR / "standard_results.csv",
    index=False
)

display(
    standard_results
)

print(
    "\nMaster standard results saved to:"
)

print(
    OUT_DIR / "standard_results.csv"
)

In [ ]:
# ============================================================
# 21C. SUFFIX HOLD-OUT MACRO-F1
# ============================================================

def plot_suffix_holdout(
    suffix_summary
):

    models = [
        "Majority-class baseline",
        "Rule-based suffix (masked)",
        "Statistical suffix",
        "Char N-gram",
        "BiLSTM"
    ]

    # Rule-Full Reference deliberately excluded from
    # the fair-comparison figure.

    data = (
        suffix_summary[
            suffix_summary["model"].isin(
                models
            )
        ]
        .copy()
    )

    pivot = data.pivot(
        index="held_out_suffix",
        columns="model",
        values="macro_f1_present_classes"
    )

    pivot = pivot.reindex(
        PRIMARY_HOLDOUT_SUFFIXES
    )

    pivot = pivot.reindex(
        columns=models
    )

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    pivot.plot(
        kind="bar",
        ax=ax
    )

    ax.set_xlabel(
        "Held-out suffix"
    )

    ax.set_ylabel(
        "Macro-F1"
    )

    ax.set_ylim(
        0,
        1
    )

    ax.set_title(
        "Morphological Generalization "
        "Across Held-Out Suffix Families"
    )

    ax.set_xticklabels(
        [
            f"-{suffix}"
            for suffix in pivot.index
        ],
        rotation=0
    )

    ax.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

    save_figure(
        fig,
        "suffix_holdout_macro_f1.png"
    )

In [ ]:
plot_suffix_holdout(
    suffix_summary
)